<a href="https://colab.research.google.com/github/Dasheeer/lab_REST-API-React/blob/main/REST_API_React_%D0%A1%D1%82%D1%83%D0%B4%D0%B5%D0%BD%D1%87%D0%B5%D1%81%D0%BA%D0%B8%D0%B9_%D0%B6%D1%83%D1%80%D0%BD%D0%B0%D0%BB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%pip install fastapi uvicorn sqlalchemy nest_asyncio pyngrok

In [3]:

import time
import threading
import nest_asyncio
import uvicorn

from fastapi import FastAPI, HTTPException, Path, Query
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel, Field, ConfigDict
from sqlalchemy import create_engine, Column, Integer, String, ForeignKey
from sqlalchemy.orm import declarative_base, sessionmaker
from pyngrok import ngrok

# Разрешаем запуск сервера в Google Colab
nest_asyncio.apply()

# 1. Создаём приложение
app = FastAPI(
    title="API студенческого журнала",
    version="1.0.0"
)
SUBJECTS = [
    "Математика",
    "Окружающий мир",
    "Литература",
    "Программная инженерия",
    "Операционные системы",
    "Компьютерные сети",
    "Информационная безопасность",
    "Английский язык",
    "ИЗО",
]


@app.get("/subjects")
def get_subjects():
    return SUBJECTS
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=False,
    allow_methods=["*"],
    allow_headers=["*"],
)
# 2. Подключаем базу данных SQLite
DATABASE_URL = "sqlite:///students.db"

engine = create_engine(
    DATABASE_URL,
    connect_args={"check_same_thread": False}
)

SessionLocal = sessionmaker(
    autocommit=False,
    autoflush=False,
    bind=engine
)

Base = declarative_base()


# 3. Модель студента
class Student(Base):
    __tablename__ = "students"

    id = Column(Integer, primary_key=True, index=True)
    name = Column(String, nullable=False)
    group_name = Column(String, nullable=False)


# 4. Модель оценки
class Grade(Base):
    __tablename__ = "grades"

    id = Column(Integer, primary_key=True, index=True)
    student_id = Column(
        Integer,
        ForeignKey("students.id"),
        nullable=False
    )
    subject = Column(String, nullable=False)
    grade = Column(Integer, nullable=False)


# Создаём таблицы, если их ещё нет
Base.metadata.create_all(bind=engine)


# 5. Схема данных студента
class StudentCreate(BaseModel):
    name: str = Field(
        min_length=1,
        description="Фамилия и имя студента",
        examples=["Анна Смирнова"]
    )
    group_name: str = Field(
        min_length=1,
        description="Название учебной группы",
        examples=["БСТ2556"]
    )

    model_config = ConfigDict(
        json_schema_extra={
            "example": {
                "name": "Анна Смирнова",
                "group_name": "БСТ2556"
            }
        }
    )


# 6. Схема данных оценки
class GradeCreate(BaseModel):
    subject: str = Field(
        min_length=1,
        description="Название учебной дисциплины",
        examples=["Базы данных"]
    )
    grade: int = Field(
        ge=2,
        le=5,
        description="Оценка по пятибалльной шкале: от 2 до 5",
        examples=[5]
    )

    model_config = ConfigDict(
        json_schema_extra={
            "example": {
                "subject": "Базы данных",
                "grade": 5
            }
        }
    )


# 7. Преобразование данных в JSON
def student_to_dict(student):
    return {
        "id": student.id,
        "name": student.name,
        "group_name": student.group_name
    }


def grade_to_dict(grade):
    return {
        "id": grade.id,
        "student_id": grade.student_id,
        "subject": grade.subject,
        "grade": grade.grade
    }


# 8. Получить список всех студентов — GET
@app.get(
    "/students",
    tags=["Студенты"],
    summary="Получить список студентов",
    description=(
        "Возвращает список студентов. "
        "Можно фильтровать по учебной группе "
        "и сортировать по имени."
    )
)
def get_students(
    group_name: str | None = Query(
        default=None,
        description="Фильтр по названию учебной группы",
        examples=["БСТ2556"]
    ),
    sort_order: str = Query(
        default="asc",
        pattern="^(asc|desc)$",
        description=(
            "Порядок сортировки по имени: "
            "asc — по алфавиту, desc — в обратном порядке"
        )
    )
):
    db = SessionLocal()
    try:
        query = db.query(Student)

        # Фильтрация по учебной группе
        if group_name:
            query = query.filter(
                Student.group_name == group_name
            )

        # Сортировка по имени
        if sort_order == "asc":
            query = query.order_by(Student.name.asc())
        else:
            query = query.order_by(Student.name.desc())

        students = query.all()

        return [student_to_dict(s) for s in students]
    finally:
        db.close()



# 9. Получить информацию о студенте — GET
@app.get(
    "/students/{student_id}",
    tags=["Студенты"],
    summary="Получить информацию о студенте",
    description="Возвращает данные студента по его идентификатору."
)
def get_student(
    student_id: int = Path(
        ...,
        description="Уникальный идентификатор студента"
    )
):
    db = SessionLocal()
    try:
        student = db.query(Student).filter(
            Student.id == student_id
        ).first()

        if student is None:
            raise HTTPException(
                status_code=404,
                detail="Студент не найден"
            )

        return student_to_dict(student)
    finally:
        db.close()


# 10. Добавить студента — POST
@app.post(
    "/students",
    tags=["Студенты"],
    status_code=201,
    summary="Добавить студента",
    description="Создаёт новую запись о студенте."
)
def create_student(data: StudentCreate):
    db = SessionLocal()
    try:
        student = Student(
            name=data.name,
            group_name=data.group_name
        )
        db.add(student)
        db.commit()
        db.refresh(student)

        return student_to_dict(student)
    finally:
        db.close()


# 11. Изменить данные студента — PUT
@app.put(
    "/students/{student_id}",
    tags=["Студенты"],
    summary="Изменить данные студента",
    description="Обновляет имя и учебную группу студента."
)
def update_student(student_id: int, data: StudentCreate):
    db = SessionLocal()
    try:
        student = db.query(Student).filter(
            Student.id == student_id
        ).first()

        if student is None:
            raise HTTPException(
                status_code=404,
                detail="Студент не найден"
            )

        student.name = data.name
        student.group_name = data.group_name

        db.commit()
        db.refresh(student)

        return student_to_dict(student)
    finally:
        db.close()


# 12. Удалить студента — DELETE
@app.delete(
    "/students/{student_id}",
    tags=["Студенты"],
    summary="Удалить студента",
    description="Удаляет студента и все его оценки."
)
def delete_student(student_id: int):
    db = SessionLocal()
    try:
        student = db.query(Student).filter(
            Student.id == student_id
        ).first()

        if student is None:
            raise HTTPException(
                status_code=404,
                detail="Студент не найден"
            )

        db.query(Grade).filter(
            Grade.student_id == student_id
        ).delete()

        db.delete(student)
        db.commit()

        return {"message": "Студент успешно удалён"}
    finally:
        db.close()


# 13. Добавить оценку студенту — POST
@app.post(
    "/students/{student_id}/grades",
    tags=["Студенты"],
    status_code=201,
    summary="Добавить оценку студенту",
    description="Добавляет студенту оценку по дисциплине."
)
def create_grade(student_id: int, data: GradeCreate):
    db = SessionLocal()
    try:
        student = db.query(Student).filter(
            Student.id == student_id
        ).first()

        if student is None:
            raise HTTPException(
                status_code=404,
                detail="Студент не найден"
            )

        grade = Grade(
            student_id=student_id,
            subject=data.subject,
            grade=data.grade
        )

        db.add(grade)
        db.commit()
        db.refresh(grade)

        return grade_to_dict(grade)
    finally:
        db.close()


# 14. Получить оценки студента — GET
@app.get(
    "/students/{student_id}/grades",
    tags=["Студенты"],
    summary="Получить оценки студента",
    description="Возвращает все оценки студента по дисциплинам."
)
def get_grades(student_id: int):
    db = SessionLocal()
    try:
        student = db.query(Student).filter(
            Student.id == student_id
        ).first()

        if student is None:
            raise HTTPException(
                status_code=404,
                detail="Студент не найден"
            )

        grades = db.query(Grade).filter(
            Grade.student_id == student_id
        ).all()

        return [grade_to_dict(g) for g in grades]
    finally:
        db.close()


# 15. Изменить оценку — PUT
@app.put(
    "/grades/{grade_id}",
    tags=["Студенты"],
    summary="Изменить оценку",
    description="Изменяет дисциплину или балл существующей оценки."
)
def update_grade(grade_id: int, data: GradeCreate):
    db = SessionLocal()
    try:
        grade = db.query(Grade).filter(
            Grade.id == grade_id
        ).first()

        if grade is None:
            raise HTTPException(
                status_code=404,
                detail="Оценка не найдена"
            )

        grade.subject = data.subject
        grade.grade = data.grade

        db.commit()
        db.refresh(grade)

        return grade_to_dict(grade)
    finally:
        db.close()


# 16. Удалить оценку — DELETE
@app.delete(
    "/grades/{grade_id}",
    tags=["Студенты"],
    summary="Удалить оценку",
    description="Удаляет выбранную оценку из журнала."
)
def delete_grade(grade_id: int):
    db = SessionLocal()
    try:
        grade = db.query(Grade).filter(
            Grade.id == grade_id
        ).first()

        if grade is None:
            raise HTTPException(
                status_code=404,
                detail="Оценка не найдена"
            )

        db.delete(grade)
        db.commit()

        return {"message": "Оценка успешно удалена"}
    finally:
        db.close()
# 17. Получить общую ведомость оценок — GET
@app.get(
    "/grades",
    tags=["Успеваемость"],
    summary="Получить общую ведомость оценок",
    description=(
        "Возвращает оценки всех студентов. "
        "Можно фильтровать по дисциплине, группе и баллу."
    )
)
def get_all_grades(
    subject: str | None = Query(
        default=None,
        description="Фильтр по названию дисциплины"
    ),
    group_name: str | None = Query(
        default=None,
        description="Фильтр по учебной группе"
    ),
    grade: int | None = Query(
        default=None,
        ge=2,
        le=5,
        description="Фильтр по оценке от 2 до 5"
    )
):
    db = SessionLocal()
    try:
        query = (
            db.query(Grade, Student)
            .join(Student, Grade.student_id == Student.id)
        )

        if subject:
            query = query.filter(Grade.subject == subject)

        if group_name:
            query = query.filter(Student.group_name == group_name)

        if grade is not None:
            query = query.filter(Grade.grade == grade)

        results = query.order_by(Student.name.asc()).all()

        return [
            {
                "id": g.id,
                "student_id": s.id,
                "name": s.name,
                "group_name": s.group_name,
                "subject": g.subject,
                "grade": g.grade
            }
            for g, s in results
        ]
    finally:
        db.close()


# 18. Получить средний балл конкретного студента — GET
@app.get(
    "/students/{student_id}/average-grade",
    tags=["Успеваемость"],
    summary="Получить средний балл студента",
    description="Возвращает средний балл студента по всем его оценкам."
)
def get_student_average_grade(student_id: int):
    db = SessionLocal()
    try:
        student = db.query(Student).filter(
            Student.id == student_id
        ).first()

        if student is None:
            raise HTTPException(
                status_code=404,
                detail="Студент не найден"
            )

        grades = db.query(Grade).filter(
            Grade.student_id == student_id
        ).all()

        average = (
            round(sum(g.grade for g in grades) / len(grades), 2)
            if grades else None
        )

        return {
            "student_id": student.id,
            "name": student.name,
            "group_name": student.group_name,
            "grades_count": len(grades),
            "average_grade": average
        }
    finally:
        db.close()


# 17. Запуск сервера
server = uvicorn.Server(
    uvicorn.Config(
        app,
        host="0.0.0.0",
        port=8002,
        log_level="warning"
    )
)

api_thread = threading.Thread(
    target=server.run,
    daemon=True
)
api_thread.start()

# Проверяем, запустился ли сервер
for _ in range(30):
    if server.started:
        break
    time.sleep(0.5)

if not server.started:
    raise RuntimeError(
        "Сервер не запустился. Возможно, порт 8002 занят. "
        "Перезапусти среду выполнения Colab и попробуй снова."
    )

# 18. Создаём публичную ссылку для Swagger
ngrok.kill()
tunnel = ngrok.connect(8002)
public_url = tunnel.public_url

print("API успешно запущен!")
print("Ссылка на API:", public_url)
print("Документация Swagger:", public_url + "/docs")
print("База данных SQLite готова к работе.")



API успешно запущен!
Ссылка на API: https://9b79-136-71-233-2.ngrok-free.app
Документация Swagger: https://9b79-136-71-233-2.ngrok-free.app/docs
База данных SQLite готова к работе.


In [1]:
from pyngrok import ngrok

ngrok.kill()

ngrok.set_auth_token("2i6MWJSxZw1l5xxiFfIm5BMBMEd_4paa8PRPN2Du82nVYnkcd")

PyngrokNgrokError: An error occurred when saving the auth token: b'Authtoken already set in configuration file: /root/.config/ngrok/ngrok.yml\n'

In [2]:
from pyngrok import ngrok

ngrok.kill()

tunnel = ngrok.connect(8002)

public_url = tunnel.public_url

print("Swagger UI:", public_url + "/docs")
print("ReDoc:", public_url + "/redoc")

Swagger UI: https://b93b-136-71-233-2.ngrok-free.app/docs
ReDoc: https://b93b-136-71-233-2.ngrok-free.app/redoc


In [4]:
import pyngrok
print("pyngrok установлен")

pyngrok установлен


In [3]:
pip install pyngrok

In [4]:

from fastapi.responses import HTMLResponse

html = r"""
<!DOCTYPE html>
<html lang="ru">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Студенческий журнал</title>
<script src="https://unpkg.com/react@18/umd/react.development.js"></script>
<script src="https://unpkg.com/react-dom@18/umd/react-dom.development.js"></script>
<script src="https://unpkg.com/@babel/standalone/babel.min.js"></script>
<style>
*{box-sizing:border-box}
body{font-family:Arial,sans-serif;background:#f3f5f9;margin:0;padding:24px;color:#20283a}
main{max-width:1050px;margin:auto;background:white;padding:26px;border-radius:14px;box-shadow:0 4px 20px #1c2b4a0b}
h1{margin-top:0}.muted{color:#667085}
nav,.actions{display:flex;gap:7px;flex-wrap:wrap;margin:16px 0}
input,select,button{padding:10px 12px;margin:4px 4px 4px 0;border:1px solid #ccd2df;border-radius:7px;font-size:14px}
input,select{max-width:100%}
button{cursor:pointer;background:#2457d6;color:white;border:0}
button:hover{opacity:.9}
button:disabled{opacity:.55;cursor:not-allowed}
button.secondary{background:#65718a}
button.danger{background:#c0392b}
button.light{background:#e9efff;color:#2457d6}
button.active{background:#173b96}
.panel{background:#f7f8fb;padding:16px;border-radius:10px;margin:14px 0}
.stats{display:flex;gap:12px;flex-wrap:wrap;margin:16px 0}
.stat{flex:1;min-width:150px;padding:16px;background:#f1f5ff;border-radius:10px}
.stat strong{display:block;font-size:25px;margin-top:6px}
table{border-collapse:collapse;width:100%;margin-top:12px}
th,td{text-align:left;border-bottom:1px solid #e4e7ee;padding:11px 8px;vertical-align:top}
th{background:#f7f8fb}
.table-wrap{overflow-x:auto}
.message,.success{padding:12px;border-radius:8px;margin:14px 0;overflow-wrap:anywhere}
.message{background:#fff0ef;color:#b42318}
.success{background:#ecfdf3;color:#067647}
.form-grid{display:grid;grid-template-columns:repeat(2,minmax(0,1fr));gap:12px}
.field{display:flex;flex-direction:column;gap:5px}
.field label{font-size:14px;font-weight:bold}
.field input,.field select{width:100%;margin:0}
.detail-row{padding:12px 0;border-bottom:1px solid #e4e7ee}
.detail-label{display:block;color:#667085;font-size:13px;margin-bottom:4px}
.empty{color:#667085;padding:18px 0}
.grade-mark{font-size:17px;font-weight:bold}
@media(max-width:650px){
  body{padding:10px}
  main{padding:16px}
  .form-grid{grid-template-columns:1fr}
}
</style>
</head>
<body>
<main><div id="root">Загрузка приложения...</div></main>

<script type="text/babel">
const {useState,useEffect}=React;

function App(){
  const [screen,setScreen]=useState("list");
  const [students,setStudents]=useState([]);
  const [search,setSearch]=useState("");
  const [group,setGroup]=useState("Все группы");
  const [form,setForm]=useState(null);
  const [details,setDetails]=useState(null);
  const [grades,setGrades]=useState([]);
  const [gradeForm,setGradeForm]=useState(null);
  const [studentAverage,setStudentAverage]=useState(null);
  const [averageLoading,setAverageLoading]=useState(false);

  const [subjectCatalog,setSubjectCatalog]=useState([]);
  const [newGradeStudent,setNewGradeStudent]=useState("");
  const [newGradeSubject,setNewGradeSubject]=useState("");
  const [newGradeValue,setNewGradeValue]=useState("5");
  const [savingGrade,setSavingGrade]=useState(false);

  const [allGrades,setAllGrades]=useState([]);
  const [subjectFilter,setSubjectFilter]=useState("Все предметы");
  const [performanceGroup,setPerformanceGroup]=useState("Все группы");
  const [performanceGrade,setPerformanceGrade]=useState("Все оценки");
  const [performanceLoading,setPerformanceLoading]=useState(false);

  const [error,setError]=useState("");
  const [success,setSuccess]=useState("");
  const [loading,setLoading]=useState(false);

  async function request(path,options={}){
    const response=await fetch(path,{
      ...options,
      headers:{
        "Content-Type":"application/json",
        "ngrok-skip-browser-warning":"true",
        ...(options.headers||{})
      }
    });

    if(!response.ok){
      const detail=await response.text();
      throw new Error("Ошибка "+response.status+": "+detail);
    }

    if(response.status===204)return null;
    return response.json();
  }

  async function loadStudents(){
    setLoading(true);
    try{
      setStudents(await request("/students"));
    }catch(e){
      setError("Не удалось загрузить студентов: "+e.message);
    }finally{
      setLoading(false);
    }
  }

  async function loadSubjects(){
    try{
      const data=await request("/subjects");
      if(!Array.isArray(data)){
        throw new Error("API должен возвращать список предметов");
      }
      setSubjectCatalog(data);
      if(data.length>0){
        setNewGradeSubject(current=>
          current&&data.includes(current)?current:data[0]
        );
      }
    }catch(e){
      setError("Не удалось загрузить предметы. Проверь GET /subjects: "+e.message);
    }
  }

  async function loadAllGrades(showError=true){
    setPerformanceLoading(true);
    try{
      setAllGrades(await request("/grades"));
    }catch(e){
      if(showError){
        setError("Не удалось загрузить общую ведомость: "+e.message);
      }
    }finally{
      setPerformanceLoading(false);
    }
  }

  useEffect(()=>{
    loadStudents();
    loadAllGrades();
    loadSubjects();
  },[]);

  const groups=[
    ...new Set(students.map(s=>s.group_name).filter(Boolean))
  ].sort((a,b)=>a.localeCompare(b,"ru"));

  const filtered=students.filter(s=>
    (s.name||"").toLowerCase().includes(search.toLowerCase())&&
    (group==="Все группы"||s.group_name===group)
  );

  const subjects=[
    ...new Set([
      ...subjectCatalog,
      ...allGrades.map(g=>g.subject).filter(Boolean)
    ])
  ].sort((a,b)=>a.localeCompare(b,"ru"));

  const performanceGroups=[
    ...new Set(allGrades.map(g=>g.group_name).filter(Boolean))
  ].sort((a,b)=>a.localeCompare(b,"ru"));

  const filteredAllGrades=allGrades.filter(g=>
    (subjectFilter==="Все предметы"||g.subject===subjectFilter)&&
    (performanceGroup==="Все группы"||g.group_name===performanceGroup)&&
    (performanceGrade==="Все оценки"||String(g.grade)===performanceGrade)
  );


  const averageGrade = filteredAllGrades.length
    ? (
        filteredAllGrades.reduce(
          (sum, g) => sum + Number(g.grade),
          0
        ) / filteredAllGrades.length
      ).toFixed(2)
    : "—";

  // Средний балл каждого студента по всем его оценкам
  const studentAverages = students.map(student => {
    const studentGrades = allGrades.filter(
      g => String(g.student_id) === String(student.id)
    );

    const average = studentGrades.length
      ? studentGrades.reduce(
          (sum, g) => sum + Number(g.grade),
          0
        ) / studentGrades.length
      : null;

    return {
      student_id: student.id,
      name: student.name,
      group_name: student.group_name,
      grades_count: studentGrades.length,
      average_grade: average
    };
  });

  // Средний балл по каждой дисциплине
  const subjectAverages = subjects
    .map(subject => {
      const subjectGrades = allGrades.filter(
        g => g.subject === subject
      );

      const average = subjectGrades.length
        ? subjectGrades.reduce(
            (sum, g) => sum + Number(g.grade),
            0
          ) / subjectGrades.length
        : null;

      return {
        subject,
        grades_count: subjectGrades.length,
        average_grade: average
      };
    })
    .filter(subject => subject.grades_count > 0);

  function go(name) {
    setError("");
    setSuccess("");
    setScreen(name);
    if (name === "performance") {
      loadAllGrades();
      loadSubjects();
    }
  }


  function go(name){
    setError("");
    setSuccess("");
    setScreen(name);
    if(name==="performance"){
      loadAllGrades();
      loadSubjects();
    }
  }

  function createStudent(){
    setError("");
    setSuccess("");
    setForm({id:null,name:"",group_name:""});
    setScreen("form");
  }

  function editStudent(s){
    setError("");
    setSuccess("");
    setForm({id:s.id,name:s.name,group_name:s.group_name});
    setScreen("form");
  }

  async function saveStudent(e){
    e.preventDefault();
    setError("");
    setSuccess("");

    const payload={
      name:form.name.trim(),
      group_name:form.group_name.trim()
    };

    if(!payload.name||!payload.group_name){
      setError("Заполни имя и учебную группу.");
      return;
    }

    try{
      if(form.id!==null){
        await request("/students/"+form.id,{
          method:"PUT",body:JSON.stringify(payload)
        });
        setSuccess("Данные студента изменены.");
      }else{
        await request("/students",{
          method:"POST",body:JSON.stringify(payload)
        });
        setSuccess("Студент добавлен.");
      }

      setForm(null);
      await loadStudents();
      await loadAllGrades(false);
      setScreen("list");
    }catch(e){
      setError("Не удалось сохранить студента: "+e.message);
    }
  }

  async function deleteStudent(id){
    if(!confirm("Удалить студента и связанные с ним оценки?"))return;
    setError("");
    setSuccess("");

    try{
      await request("/students/"+id,{method:"DELETE"});
      if(details&&details.id===id){
        setDetails(null);
        setGrades([]);
        setStudentAverage(null);
        setGradeForm(null);
      }
      await loadStudents();
      await loadAllGrades(false);
      setSuccess("Студент удалён.");
    }catch(e){
      setError("Не удалось удалить студента: "+e.message);
    }
  }

  // Загружаем карточку, оценки и средний балл одновременно.
  async function openDetails(id){
    setScreen("details");
    setDetails(null);
    setGrades([]);
    setStudentAverage(null);
    setAverageLoading(true);
    setGradeForm(null);
    setError("");
    setSuccess("");

    try{
      const [student,gradesData,averageData]=await Promise.all([
        request("/students/"+id),
        request("/students/"+id+"/grades"),
        request("/students/"+id+"/average-grade")
      ]);

      setDetails(student);
      setGrades(gradesData);
      setStudentAverage(averageData);
    }catch(e){
      setError("Не удалось загрузить данные студента: "+e.message);
    }finally{
      setAverageLoading(false);
    }
  }

  // Обновляем данные карточки после изменений оценок.
  async function refreshStudentDetails(){
    if(!details)return;

    const id=details.id;
    try{
      const [student,gradesData,averageData]=await Promise.all([
        request("/students/"+id),
        request("/students/"+id+"/grades"),
        request("/students/"+id+"/average-grade")
      ]);

      setDetails(student);
      setGrades(gradesData);
      setStudentAverage(averageData);
    }catch(e){
      setError("Не удалось обновить карточку студента: "+e.message);
    }
  }

  function createGrade(){
    setError("");
    setSuccess("");
    setGradeForm({
      id:null,
      subject:subjectCatalog[0]||"",
      grade:""
    });
  }

  function editGrade(g){
    setError("");
    setSuccess("");
    setGradeForm({
      id:g.id,
      subject:g.subject,
      grade:String(g.grade)
    });
  }

  async function saveGrade(e){
    e.preventDefault();
    setError("");
    setSuccess("");
    if(!details)return;

    const subject=gradeForm.subject.trim();
    const grade=Number(gradeForm.grade);

    if(!subject){
      setError("Выбери предмет.");
      return;
    }

    if(!Number.isInteger(grade)||grade<2||grade>5){
      setError("Оценка должна быть целым числом от 2 до 5.");
      return;
    }

    try{
      const payload={subject,grade};

      if(gradeForm.id!==null){
        await request("/grades/"+gradeForm.id,{
          method:"PUT",body:JSON.stringify(payload)
        });
        setSuccess("Оценка изменена.");
      }else{
        await request("/students/"+details.id+"/grades",{
          method:"POST",body:JSON.stringify(payload)
        });
        setSuccess("Оценка добавлена.");
      }

      setGradeForm(null);
      await refreshStudentDetails();
      await loadAllGrades(false);
    }catch(e){
      setError("Не удалось сохранить оценку: "+e.message);
    }
  }

  async function deleteGrade(id){
    if(!confirm("Удалить эту оценку?"))return;
    setError("");
    setSuccess("");

    try{
      await request("/grades/"+id,{method:"DELETE"});
      setGradeForm(null);
      await refreshStudentDetails();
      await loadAllGrades(false);
      setSuccess("Оценка удалена.");
    }catch(e){
      setError("Не удалось удалить оценку: "+e.message);
    }
  }

  async function addGradeFromPerformance(){
    setError("");
    setSuccess("");

    if(!newGradeStudent){
      setError("Выбери студента.");
      return;
    }
    if(!newGradeSubject){
      setError("Выбери предмет.");
      return;
    }

    setSavingGrade(true);
    try{
      await request(`/students/${newGradeStudent}/grades`,{
        method:"POST",
        body:JSON.stringify({
          subject:newGradeSubject,
          grade:Number(newGradeValue)
        })
      });

      await loadAllGrades(false);

      if(details&&String(details.id)===String(newGradeStudent)){
        await refreshStudentDetails();
      }

      setSuccess("Оценка успешно добавлена.");
    }catch(e){
      setError("Не удалось добавить оценку: "+e.message);
    }finally{
      setSavingGrade(false);
    }
  }

  function resetPerformanceFilters(){
    setSubjectFilter("Все предметы");
    setPerformanceGroup("Все группы");
    setPerformanceGrade("Все оценки");
  }

  return <div>
    <h1>Студенческий журнал</h1>
    <p className="muted">Управление студентами и успеваемостью через REST API</p>

    <nav>
      <button className={screen==="list"?"active":"secondary"} onClick={()=>go("list")}>1. Список студентов</button>
      <button className={screen==="form"?"active":"secondary"} onClick={createStudent}>2. Добавить студента</button>
      <button className={screen==="details"?"active":"secondary"} onClick={()=>go("details")}>3. Поиск и детали</button>
      <button className={screen==="performance"?"active":"secondary"} onClick={()=>go("performance")}>4. Успеваемость</button>
    </nav>

    {error&&<div className="message">{error}</div>}
    {success&&<div className="success">{success}</div>}

    {screen==="list"&&<section>
      <h2>Список студентов</h2>
      <div className="stats">
        <div className="stat">Всего студентов<strong>{students.length}</strong></div>
        <div className="stat">Учебных групп<strong>{groups.length}</strong></div>
      </div>
      <button className="secondary" onClick={loadStudents}>Обновить список</button>

      {loading?<p>Загрузка...</p>:<div className="table-wrap">
        <table>
          <thead><tr><th>ID</th><th>Имя</th><th>Группа</th><th>Действия</th></tr></thead>
          <tbody>
            {students.map(s=><tr key={s.id}>
              <td>{s.id}</td>
              <td>{s.name}</td>
              <td>{s.group_name}</td>
              <td><div className="actions">
                <button className="light" onClick={()=>openDetails(s.id)}>Подробнее</button>
                <button onClick={()=>editStudent(s)}>Изменить</button>
                <button className="danger" onClick={()=>deleteStudent(s.id)}>Удалить</button>
              </div></td>
            </tr>)}
            {students.length===0&&<tr><td colSpan="4" className="empty">Студентов пока нет.</td></tr>}
          </tbody>
        </table>
      </div>}
    </section>}

    {screen==="form"&&<section>
      <h2>{form&&form.id!==null?"Редактирование студента":"Добавление студента"}</h2>
      {form&&<form className="panel" onSubmit={saveStudent}>
        <div className="form-grid">
          <div className="field">
            <label>Имя студента</label>
            <input required value={form.name} placeholder="Имя и фамилия" onChange={e=>setForm({...form,name:e.target.value})}/>
          </div>
          <div className="field">
            <label>Учебная группа</label>
            <input required value={form.group_name} placeholder="Например, БСТ2556" onChange={e=>setForm({...form,group_name:e.target.value})}/>
          </div>
        </div>
        <div className="actions">
          <button type="submit">Сохранить</button>
          <button type="button" className="secondary" onClick={()=>go("list")}>Отмена</button>
        </div>
      </form>}
    </section>}

    {screen==="details"&&<section>
      <h2>Поиск и подробная информация</h2>
      <div className="panel">
        <div className="form-grid">
          <div className="field">
            <label>Поиск по имени</label>
            <input value={search} placeholder="Введи имя студента" onChange={e=>setSearch(e.target.value)}/>
          </div>
          <div className="field">
            <label>Фильтр по группе</label>
            <select value={group} onChange={e=>setGroup(e.target.value)}>
              <option>Все группы</option>
              {groups.map(g=><option key={g}>{g}</option>)}
            </select>
          </div>
        </div>

        <p>Найдено студентов: <strong>{filtered.length}</strong></p>

        <div className="table-wrap">
          <table>
            <thead><tr><th>ID</th><th>Имя</th><th>Группа</th><th></th></tr></thead>
            <tbody>
              {filtered.map(s=><tr key={s.id}>
                <td>{s.id}</td>
                <td>{s.name}</td>
                <td>{s.group_name}</td>
                <td><button className="light" onClick={()=>openDetails(s.id)}>Открыть</button></td>
              </tr>)}
              {filtered.length===0&&<tr><td colSpan="4" className="empty">Ничего не найдено.</td></tr>}
            </tbody>
          </table>
        </div>
      </div>

      {details&&<div className="panel">
        <h3>Карточка студента</h3>

        <div className="detail-row">
          <span className="detail-label">ID</span>
          <strong>{details.id}</strong>
        </div>
        <div className="detail-row">
          <span className="detail-label">Имя</span>
          <strong>{details.name}</strong>
        </div>
        <div className="detail-row">
          <span className="detail-label">Группа</span>
          <strong>{details.group_name}</strong>
        </div>

        <div className="detail-row">
          <span className="detail-label">Средний балл</span>
          <strong>
            {averageLoading
              ?"Загрузка..."
              :studentAverage===null
                ?"—"
                :studentAverage.average_grade===null
                  ?"Пока нет оценок"
                  :Number(studentAverage.average_grade).toFixed(2)}
          </strong>
          {studentAverage!==null&&studentAverage.average_grade!==null&&
            <span className="detail-label">
              Всего оценок: {studentAverage.grades_count}
            </span>}
        </div>

        <div className="actions">
          <button onClick={()=>editStudent(details)}>Редактировать студента</button>
          <button className="secondary" onClick={createGrade}>+ Добавить оценку</button>
        </div>

        <h3>Оценки по дисциплинам</h3>
        {grades.length>0?<div className="table-wrap">
          <table>
            <thead><tr><th>Дисциплина</th><th>Оценка</th><th>Действия</th></tr></thead>
            <tbody>
              {grades.map(g=><tr key={g.id}>
                <td>{g.subject}</td>
                <td><strong>{g.grade}</strong></td>
                <td><div className="actions">
                  <button onClick={()=>editGrade(g)}>Изменить</button>
                  <button className="danger" onClick={()=>deleteGrade(g.id)}>Удалить</button>
                </div></td>
              </tr>)}
            </tbody>
          </table>
        </div>:<p className="empty">Оценок пока нет. Нажми «Добавить оценку», чтобы создать первую запись.</p>}

        {gradeForm&&<form className="panel" onSubmit={saveGrade}>
          <h3>{gradeForm.id!==null?"Редактирование оценки":"Новая оценка"}</h3>
          <div className="form-grid">
            <div className="field">
              <label>Дисциплина</label>
              <select required value={gradeForm.subject} onChange={e=>setGradeForm({...gradeForm,subject:e.target.value})}>
                <option value="">Выбери предмет</option>
                {subjects.map(s=><option key={s} value={s}>{s}</option>)}
              </select>
            </div>
            <div className="field">
              <label>Оценка (от 2 до 5)</label>
              <select required value={gradeForm.grade} onChange={e=>setGradeForm({...gradeForm,grade:e.target.value})}>
                <option value="">Выбери оценку</option>
                <option value="2">2 — неудовлетворительно</option>
                <option value="3">3 — удовлетворительно</option>
                <option value="4">4 — хорошо</option>
                <option value="5">5 — отлично</option>
              </select>
            </div>
          </div>
          <div className="actions">
            <button type="submit">Сохранить оценку</button>
            <button type="button" className="secondary" onClick={()=>setGradeForm(null)}>Отмена</button>
          </div>
        </form>}
      </div>}
    </section>}

    {screen==="performance"&&<section>
      <h2>Общая ведомость успеваемости</h2>
      <p className="muted">Добавляй оценки и просматривай результаты всех студентов.</p>

      <div className="panel">
        <h3>Добавить оценку</h3>
        <div className="form-grid">
          <div className="field">
            <label>Студент</label>
            <select value={newGradeStudent} onChange={e=>setNewGradeStudent(e.target.value)}>
              <option value="">Выбери студента</option>
              {students.map(s=><option key={s.id} value={s.id}>{s.name} — {s.group_name}</option>)}
            </select>
          </div>
          <div className="field">
            <label>Предмет</label>
            <select value={newGradeSubject} onChange={e=>setNewGradeSubject(e.target.value)}>
              <option value="">Выбери предмет</option>
              {subjectCatalog.map(s=><option key={s} value={s}>{s}</option>)}
            </select>
          </div>
          <div className="field">
            <label>Оценка</label>
            <select value={newGradeValue} onChange={e=>setNewGradeValue(e.target.value)}>
              <option value="2">2 — неудовлетворительно</option>
              <option value="3">3 — удовлетворительно</option>
              <option value="4">4 — хорошо</option>
              <option value="5">5 — отлично</option>
            </select>
          </div>
        </div>
        <div className="actions">
          <button onClick={addGradeFromPerformance} disabled={savingGrade||students.length===0||subjectCatalog.length===0}>
            {savingGrade?"Сохранение...":"Добавить оценку"}
          </button>
          <button className="secondary" onClick={()=>{loadSubjects();loadStudents();}}>Обновить списки</button>
        </div>
        {students.length===0&&<p className="empty">Сначала добавь хотя бы одного студента.</p>}
        {subjectCatalog.length===0&&<p className="empty">Список предметов пуст. Проверь, что GET /subjects возвращает список.</p>}
      </div>

      <div className="stats">
        <div className="stat">Всего оценок<strong>{allGrades.length}</strong></div>
        <div className="stat">Результатов по фильтру<strong>{filteredAllGrades.length}</strong></div>
        <div className="stat">Средний балл выборки<strong>{averageGrade}</strong></div>
      </div>

{/* Средний балл по студентам */}
<div className="panel">
  <h3>Средний балл по студентам</h3>
  <div className="table-wrap">
    <table>
      <thead>
        <tr>
          <th>Студент</th>
          <th>Группа</th>
          <th>Количество оценок</th>
          <th>Средний балл</th>
        </tr>
      </thead>
      <tbody>
        {studentAverages.map(student => (
          <tr key={student.student_id}>
            <td>{student.name}</td>
            <td>{student.group_name}</td>
            <td>{student.grades_count}</td>
            <td>
              {student.average_grade === null
                ? "Нет оценок"
                : student.average_grade.toFixed(2)}
            </td>
          </tr>
        ))}
        {studentAverages.length === 0 && (
          <tr>
            <td colSpan="4" className="empty">
              Студентов пока нет.
            </td>
          </tr>
        )}
      </tbody>
    </table>
  </div>
</div>

{/* Средний балл по предметам */}
<div className="panel">
  <h3>Средний балл по предметам</h3>
  <div className="table-wrap">
    <table>
      <thead>
        <tr>
          <th>Предмет</th>
          <th>Количество оценок</th>
          <th>Средний балл</th>
        </tr>
      </thead>
      <tbody>
        {subjectAverages.map(subject => (
          <tr key={subject.subject}>
            <td>{subject.subject}</td>
            <td>{subject.grades_count}</td>
            <td>{subject.average_grade.toFixed(2)}</td>
          </tr>
        ))}
        {subjectAverages.length === 0 && (
          <tr>
            <td colSpan="3" className="empty">
              Оценок по предметам пока нет.
            </td>
          </tr>
        )}
      </tbody>
    </table>
  </div>
</div>

      <div className="panel">
        <h3>Фильтры ведомости</h3>
        <div className="form-grid">
          <div className="field">
            <label>Учебная дисциплина</label>
            <select value={subjectFilter} onChange={e=>setSubjectFilter(e.target.value)}>
              <option>Все предметы</option>
              {subjects.map(s=><option key={s} value={s}>{s}</option>)}
            </select>
          </div>
          <div className="field">
            <label>Учебная группа</label>
            <select value={performanceGroup} onChange={e=>setPerformanceGroup(e.target.value)}>
              <option>Все группы</option>
              {performanceGroups.map(g=><option key={g} value={g}>{g}</option>)}
            </select>
          </div>
          <div className="field">
            <label>Оценка</label>
            <select value={performanceGrade} onChange={e=>setPerformanceGrade(e.target.value)}>
              <option value="Все оценки">Все оценки</option>
              <option value="2">2 — неудовлетворительно</option>
              <option value="3">3 — удовлетворительно</option>
              <option value="4">4 — хорошо</option>
              <option value="5">5 — отлично</option>
            </select>
          </div>
        </div>
        <div className="actions">
          <button onClick={()=>loadAllGrades()}>Обновить ведомость</button>
          <button className="secondary" onClick={resetPerformanceFilters}>Сбросить фильтры</button>
        </div>
      </div>

      {performanceLoading?<p>Загрузка ведомости...</p>:<div className="table-wrap">
        <table>
          <thead><tr><th>Студент</th><th>Группа</th><th>Дисциплина</th><th>Оценка</th></tr></thead>
          <tbody>
            {filteredAllGrades.map(g=><tr key={g.id}>
              <td><button className="light" onClick={()=>openDetails(g.student_id)}>{g.name}</button></td>
              <td>{g.group_name}</td>
              <td>{g.subject}</td>
              <td className="grade-mark">{g.grade}</td>
            </tr>)}
            {filteredAllGrades.length===0&&<tr><td colSpan="4" className="empty">
              {allGrades.length===0
                ?"Оценок пока нет. Добавь первую оценку в форме выше."
                :"По выбранным фильтрам ничего не найдено."}
            </td></tr>}
          </tbody>
        </table>
      </div>}
    </section>}
  </div>;
}

ReactDOM.createRoot(document.getElementById("root")).render(<App />);
</script>
</body>
</html>
"""

@app.get("/", response_class=HTMLResponse)
def student_journal_page():
    return HTMLResponse(content=html)
